# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nhatle16/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
!git clone https://github.com/nhatle16/flyrank-ml-internship.git

# Loading the dataset
import pandas as pd, numpy as np
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I will frame my Ranking Signal Analysis lane as a classification problem. The goal is to determine whether observable content and search signals can distinguish between pages with stronger and weaker observed search performance.

For the initial experiment, I will define two groups based on observed average search position: pages with an average position of 10 or better, and pages with an average position worse than 10. The model will use signals such as content characteristics, search volume, competition, content age, and engagement to learn patterns associated with these two groups.

I chose classification because it provides a clear and measurable outcome while allowing multiple signals to be considered together. The goal is to investigate whether the available signals contain useful information about observed search performance.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# I organized signals in the dataset into groups
signal_groups = {
    "Content characteristics": [
        "content_type",
        "main_intent",
        "word_count",
        "char_count",
        "content_age_days",
        "days_since_last_update",
    ],
    "Search characteristics": [
        "search_volume",
        "competition",
        "competition_level",
        "cpc",
    ],
    "Performance measures": [
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "scroll_rate",
    ],
}

for group, columns in signal_groups.items():
    available = [col for col in columns if col in df.columns]

    print(f"\n{group} ({len(available)} columns)")
    print(available)


Content characteristics (6 columns)
['content_type', 'main_intent', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update']

Search characteristics (4 columns)
['search_volume', 'competition', 'competition_level', 'cpc']

Performance measures (7 columns)
['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I would predict whether a content observation belongs to the group with an observed average search position of 10 or better.

The underlying outcome, which is `avg_position`, is an observed measurement in the dataset. I will convert this continuous value into a binary label using a defined threshold:

- `1` if `avg_position <= 10`
- `0` if `avg_position > 10`

Therefore, the label is not directly provided as a classification label in the dataset. It is derived from an observed outcome using a predefined rule.

I use this derived label as a proxy for stronger observed search visibility. It does not represent Google's internal ranking decision and should not be interpreted as causal evidence about ranking.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Create the binary target
df["position_target"] = (df["avg_position"] <= 10).astype(int)

# Show the target distribution
target_summary = (
    df["position_target"]
    .value_counts()
    .rename(index={0: "avg_position > 10", 1: "avg_position <= 10"})
    .to_frame("count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(df) * 100
).round(2)

display(target_summary)

,count,percentage
position_target,,
avg_position > 10,15812,52.71
avg_position <= 10,14188,47.29


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use **ROC-AUC** as the primary evaluation metric because this is a binary classification problem. ROC-AUC measures how well the model distinguishes observations with an average position of 10 or better from those with a worse observed average position.

I will not define an arbitrary threshold such as "ROC-AUC above 0.80 is good." Instead, I will compare the ML model against a simple baseline and consider the model useful only if it provides meaningful improvement over that baseline.

As a practical goal, I would look for a ROC-AUC score clearly above 0.50, since 0.50 represents random discrimination. The stronger the improvement over the baseline, the more evidence there is that the available signals contain useful predictive information.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Create the binary position target
df["position_target"] = (df["avg_position"] <= 10).astype(int)

# The cardinality of each group
print(df["position_target"].value_counts())

# The proportion of each group
print(
    df["position_target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

position_target
0    15812
1    14188
Name: count, dtype: int64
position_target
0    52.71
1    47.29
Name: proportion, dtype: float64


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis for my Ranking Signal Analysis lane is one content/page observation.

Each row represents a piece of content identified by `content_id`, along with its content characteristics, search-related attributes, and measured search-performance metrics.

I will verify whether each `content_id` corresponds to one row before treating the row as the unit of analysis.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the unit of analysis
print(f"Total rows: {len(df):,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Duplicate content IDs: {df['content_id'].duplicated().sum():,}")

# THERE ARE NO DUPLICATES, ONE ROW REPRESENTS A SINGLE UNIQUE OBSERVATION => ONE ROW IS A UNIT OF ANALYSIS

Total rows: 30,000
Unique content IDs: 30,000
Duplicate content IDs: 0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pattern may be too messy for a simple if-statement because search performance is represented by multiple signals rather than a single content characteristic. The dataset contains variables such as search volume, competition, content age, word count, search intent, engagement rate, and scroll rate.

A fixed rule would require manually choosing thresholds, for example "if word count > X, prioritize the page." Such a rule would only consider a small part of the available information and may miss interactions between multiple signals.

I therefore want to test whether a model can combine several signals and identify patterns that are difficult to capture with a small set of manually defined rules. I will compare the ML approach against a simple baseline rather than assuming that ML is automatically better.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Potential features
candidate_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate",
]

# Check that these features exist in the dataset
available_features = [
    col for col in candidate_features
    if col in df.columns
]

print(f"Number of candidate signals: {len(available_features)}")
print("\nCandidate signals:")
for feature in available_features:
    print(f"- {feature}")

Number of candidate signals: 9

Candidate signals:
- search_volume
- competition
- cpc
- word_count
- char_count
- content_age_days
- days_since_last_update
- engagement_rate
- scroll_rate


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.